In [1]:
import pandas as pd
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, callbacks
from sklearn.model_selection import train_test_split

In [2]:
# ────────────────────────────────────────────────────────────────────────────────
# 0) Hyperparameters & Constants
# ────────────────────────────────────────────────────────────────────────────────
MAX_VOCAB_SIZE    = 20_000
MAX_SEQUENCE_LEN  = 200
EMBEDDING_DIM     = 128
LSTM_UNITS        = 64
BATCH_SIZE        = 64
EPOCHS            = 1
AUTOTUNE          = tf.data.AUTOTUNE
NUM_CLASSES       = 4
CLASS_NAMES       = ["World", "Sports", "Business", "Sci/Tech"]


In [3]:
train_path = "/Users/sameerkhan/Desktop/sameerkhan/data/nlp/ag_news/train.csv"
test_path = "/Users/sameerkhan/Desktop/sameerkhan/data/nlp/ag_news/test.csv"

MODEL_DIR = "/Users/sameerkhan/Desktop/sameerkhan/weights/nlp/bilstm"
os.makedirs(MODEL_DIR, exist_ok=True)

CHECKPOINT_FILE = f"agnews_bilstm_fun_1.h5"

FINAL_MODEL_FILE = "agnews_bilstm_fun_1.keras"

VOCAB_FILE = MODEL_DIR + "/agnews_vocab.txt"

In [4]:
train_df = pd.read_csv(train_path,header=0)
train_df = train_df.rename(columns={"Class Index":"label","Title":"title","Description":"description"})

test_df = pd.read_csv(test_path,header=0)
test_df = test_df.rename(columns={"Class Index":"label","Title":"title","Description":"description"})


# zero-based labels
train_df["label"] = train_df["label"].astype(int) - 1
test_df["label"]  = test_df["label"].astype(int) - 1

# combine title + description
train_df["text"] = train_df["title"] + " " + train_df["description"]
test_df["text"]  = test_df["title"]  + " " + test_df["description"]

In [5]:
# ────────────────────────────────────────────────────────────────────────────────
# 2) Train/validation split
# ────────────────────────────────────────────────────────────────────────────────
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df["text"].values,
    train_df["label"].values,
    test_size=0.2,
    random_state=42,
    stratify=train_df["label"].values
)
test_texts  = test_df["text"].values
test_labels = test_df["label"].values


In [6]:
# ────────────────────────────────────────────────────────────────────────────────
# 3) TextVectorization
# ────────────────────────────────────────────────────────────────────────────────
vectorizer = layers.TextVectorization(
    max_tokens=MAX_VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=MAX_SEQUENCE_LEN
)
vectorizer.adapt(train_texts)

def vectorize_text(text, label):
    text = tf.expand_dims(text, -1)
    token_ids = vectorizer(text)
    return tf.squeeze(token_ids, axis=0), label

def make_dataset(texts, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((texts, labels))
    if shuffle:
        ds = ds.shuffle(len(texts), seed=42)
    ds = ds.map(vectorize_text, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_texts, train_labels, shuffle=True)
val_ds   = make_dataset(val_texts,   val_labels)
test_ds  = make_dataset(test_texts,  test_labels)

In [7]:
# ────────────────────────────────────────────────────────────────────────────────
# 4) Build BiLSTM Functional Model
# ────────────────────────────────────────────────────────────────────────────────
inp = layers.Input(shape=(MAX_SEQUENCE_LEN,), dtype="int32", name="input_tokens")
x   = layers.Embedding(
          input_dim=MAX_VOCAB_SIZE,
          output_dim=EMBEDDING_DIM,
          input_length=MAX_SEQUENCE_LEN,
          mask_zero=True
      )(inp)
x   = layers.Bidirectional(layers.LSTM(LSTM_UNITS))(x)
x   = layers.Dropout(0.5)(x)
x   = layers.Dense(64, activation="relu")(x)
x   = layers.Dropout(0.5)(x)
out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = Model(inputs=inp, outputs=out, name="bilstm_agnews")

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

/Users/sameerkhan/Desktop/sameerkhan/venv/lib/python3.9/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "bilstm_agnews"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_tokens        │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 200, 128)  │  2,560,000 │ input_tokens[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 200)       │          0 │ input_tokens[0][… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 128)       │     98,816 │ embedding[0][0],  │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      8,256 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 4)         │        260 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,667,332 (10.18 MB)

 Trainable params: 2,667,332 (10.18 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
# ────────────────────────────────────────────────────────────────────────────────
# 5) Train
# ────────────────────────────────────────────────────────────────────────────────
ckpt = callbacks.ModelCheckpoint(
    filepath=os.path.join(MODEL_DIR, CHECKPOINT_FILE),
    monitor="val_accuracy",
    save_best_only=True
)
es = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[ckpt, es]
)

1500/1500 ━━━━━━━━━━━━━━━━━━━━ 0s 241ms/step - accuracy: 0.7885 - loss: 0.5487

1500/1500 ━━━━━━━━━━━━━━━━━━━━ 381s 253ms/step - accuracy: 0.7886 - loss: 0.5486 - val_accuracy: 0.9173 - val_loss: 0.2499


In [9]:

# ────────────────────────────────────────────────────────────────────────────────
# 6) Evaluate
# ────────────────────────────────────────────────────────────────────────────────
loss, acc = model.evaluate(test_ds)
print(f"Test accuracy: {acc:.4f}")

119/119 ━━━━━━━━━━━━━━━━━━━━ 5s 42ms/step - accuracy: 0.9074 - loss: 0.2781
Test accuracy: 0.9162


In [10]:

# ────────────────────────────────────────────────────────────────────────────────
# 7) Demo Predictions
# ────────────────────────────────────────────────────────────────────────────────
def predict(text):
    seq = vectorizer(tf.constant([text]))
    probs = model.predict(seq)[0]
    idx = int(tf.argmax(probs))
    return CLASS_NAMES[idx], float(probs[idx])

examples = [
    "NASA plans a new mission to Mars.",
    "The stock market experienced a sharp decline today."
]
for ex in examples:
    cls, conf = predict(ex)
    print(f"{cls} ({conf:.1%}): {ex}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step
Sci/Tech (99.3%): NASA plans a new mission to Mars.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
Business (83.8%): The stock market experienced a sharp decline today.
